<a href="https://colab.research.google.com/github/Johnal96/ITAI2373-NewsBot-Midterm/blob/main/NewsBot-Midterm-Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!pip install kaggle

from google.colab import files
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c learn-ai-bbc\ \(2\).zip
!unzip learn-ai-bbc\ \(2\).zip

Saving BBC News Train.csv to BBC News Train (1).csv
Saving BBC News Sample Solution.csv to BBC News Sample Solution (1).csv
Saving BBC News Test.csv to BBC News Test (1).csv
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
Archive:  learn-ai-bbc (2).zip
replace BBC News Sample Solution.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BBC News Sample Solution.csv  
replace BBC News Test.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BBC News Test.csv       
replace BBC News Train.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: BBC News Train.csv      


In [26]:
!pip install nltk spacy scikit-learn

!python -m spacy download en_core_web_sm

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
import string

import spacy
nlp = spacy.load("en_core_web_sm")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 72.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:

df = pd.read_csv("BBC News Train.csv")

print("Columns:", df.columns)
df.head()

Columns: Index(['ArticleId', 'Text', 'Category'], dtype='object')


,ArticleId,Text,Category
0,1833,worldcom ex-boss launches defence lawyers defe...,business
1,154,german business confidence slides german busin...,business
2,1101,bbc poll indicates economic gloom citizens in ...,business
3,1976,lifestyle governs mobile choice faster bett...,tech
4,917,enron bosses in $168m payout eighteen former e...,business


In [28]:
df = df[['content', 'Category']]

df = df.dropna()

print("Dataset shape:", df.shape)
df.head()

KeyError: "['content'] not in index"

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(y=df['Category'], order=df['Category'].value_counts().index)
plt.title("Category Distribution")
plt.show()

In [ ]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

df['cleaned_content'] = df['content'].apply(clean_text)

df.head()

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['cleaned_content'])
y = df['Category']

print("Feature matrix shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

sample_text = df['content'].iloc[0]
print(sample_text)
print("\nEntities:")
print(extract_entities(sample_text))

In [ ]:
from textblob import TextBlob

def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df['sentiment'] = df['content'].apply(get_sentiment)

df[['content', 'sentiment']].head()

In [ ]:
def analyze_article(article_text):
    cleaned = clean_text(article_text)

    vect = vectorizer.transform([cleaned])

    category = model.predict(vect)[0]

    entities = extract_entities(article_text)

    sentiment = get_sentiment(article_text)

    return {
        "Category": category,
        "Entities": entities,
        "Sentiment": sentiment
    }

new_article = """
Apple announced a new iPhone today in California. The CEO said it will revolutionize the market.
"""

result = analyze_article(new_article)

print(result)